[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# FTS5Model and SearchField &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: five books, the index over them, and the FTS5 check. Run it
first. Task 4 adds a row and rebuilds the index, so run these in order.


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import CharField, Model, OperationalError, SqliteDatabase, TextField
from playhouse.sqlite_ext import FTS5Model, RowIDField, SearchField

SHELF = [                                                           # title, description, shelf mark
    ("The Sea, the Sea", "The sea, the sea, and a house above the sea. A sea story.", "A1"),
    ("A Quiet Engine", "Engines, quiet ones, and one mention of the sea.", "B2"),
    ("Tides", "The sea at night, tides, and the sea again by morning.", "A1"),
    ("Stone and Slate", "Granite, slate and quarries. No water at all.", "C3"),
    ("Learning C++", "Templates, memory and the standard library.", "D4"),
]

db = SqliteDatabase(":memory:", pragmas={"foreign_keys": 1})        # SQLite enforces nothing without this


print("peewee", peewee.__version__, "| FTS5 available:", FTS5Model.fts5_installed())
print("the compile option:",
      db.execute_sql("SELECT 1 FROM pragma_compile_options() "
                     "WHERE compile_options = 'ENABLE_FTS5'").fetchone() is not None)


class Book(Model):
    """The real table, where the text lives."""

    title = CharField(max_length=80)
    blurb = TextField()
    shelf = CharField(max_length=10)

    class Meta:
        database = db


class BookIndex(FTS5Model):
    """The virtual table that searches Book, holding no copy of the text itself."""

    rowid = RowIDField()
    title = SearchField()
    blurb = SearchField()
    shelf = SearchField(unindexed=True)

    class Meta:
        database = db
        options = {"content": Book}


db.create_tables([Book, BookIndex])
for title, blurb, shelf in SHELF:
    Book.create(title=title, blurb=blurb, shelf=shelf)
BookIndex.rebuild()

print("books:", Book.select().count(), "| indexed:", BookIndex.select().count())


peewee 4.5.1 | FTS5 available: True
the compile option: True
books: 5 | indexed: 5


**1.** The same word, two ways.


In [2]:
def by_index(word):
    return [row.title for row in
            BookIndex.select().where(BookIndex.match(BookIndex.web_query(word)))]


def by_scan(word):
    return [row.title for row in Book.select().where(Book.blurb.contains(word))]


for word in ("granite", "granit", "stone"):
    print(f"  match({word!r}):    {by_index(word)}")
    print(f"  contains({word!r}): {by_scan(word)}")


  match('granite'):    ['Stone and Slate']
  contains('granite'): ['Stone and Slate']
  match('granit'):    []
  contains('granit'): ['Stone and Slate']
  match('stone'):    ['Stone and Slate']
  contains('stone'): []


Three words, three different lessons. On `granite` the two agree, and note that `contains` matched
a capital G, because the `LIKE` behind it ignores case for ASCII. On `granit` they part company: the
index holds whole words and finds nothing, while a substring test finds the row. On `stone` they
part company the other way, because the index covers the title as well as the blurb and `contains`
was only ever asked about the blurb.

That last one is the difference worth keeping: a search over several columns is one query against
the index, and doing it with `contains` means naming each column and joining the conditions
yourself. The index also answers without reading the rows, and `contains` reads all of them.


**2.** Three things a person might type.


In [3]:
for entry in ("tides OR granite", '"the sea"', "sea AND OR"):
    translated = BookIndex.web_query(entry)
    found = [row.title for row in BookIndex.select().where(BookIndex.match(translated))]
    print(f"  {entry!r:<20} -> {translated!r}")
    print(f"  {'':<20}    {found}")


  'tides OR granite'   -> '"tides" OR "granite"'
                          ['Tides', 'Stone and Slate']
  '"the sea"'          -> '"the sea"'
                          ['The Sea, the Sea', 'A Quiet Engine', 'Tides']
  'sea AND OR'         -> '"sea"'
                          ['The Sea, the Sea', 'A Quiet Engine', 'Tides']


The first keeps its `OR`, because that was meant. The second stays a phrase. The third had two
operators and nothing to join, and came back as the one word that was really in it.


**3.** Real rows, ranked.


In [4]:
results = (Book.select()
               .join(BookIndex, on=(Book.id == BookIndex.rowid))
               .where(BookIndex.match(BookIndex.web_query("sea")))
               .order_by(BookIndex.bm25()))

for position, book in enumerate(results, start=1):
    print(f"  {position}. {book.title:<18} shelf {book.shelf}")


  1. The Sea, the Sea   shelf A1
  2. Tides              shelf A1
  3. A Quiet Engine     shelf B2


The join is on `rowid`, which is the index's link back to the content table. Without it these would
be index rows, which have no shelf mark to print.


**4.** A book the index has not heard of.


In [5]:
Book.create(title="Salt and Weather", blurb="Storms, salt and the weather at sea.", shelf="A3")

before = [row.title for row in
          BookIndex.select().where(BookIndex.match(BookIndex.web_query("storms")))]
BookIndex.rebuild()
after = [row.title for row in
         BookIndex.select().where(BookIndex.match(BookIndex.web_query("storms")))]

print("in the real table:", Book.select().where(Book.title == "Salt and Weather").count())
print("before rebuild:", before)
print("after rebuild: ", after)


in the real table: 1
before rebuild: []
after rebuild:  ['Salt and Weather']


The index is a snapshot of the words as of the last `rebuild`. Writing to `book` does not touch it,
and nothing warns you, so a search simply does not find a row that is plainly there.


**5.** The same search, raising and not raising.


In [6]:
entry = "sea -"

try:
    BookIndex.select().where(BookIndex.match(entry)).count()
except OperationalError as error:
    print("match:    peewee.OperationalError:", error)

print("web_query:", repr(BookIndex.web_query(entry)))
print("found:   ", [row.title for row in
                    BookIndex.select().where(BookIndex.match(BookIndex.web_query(entry)))])


match:    peewee.OperationalError: fts5: syntax error near ""
web_query: '"sea"'
found:    ['The Sea, the Sea', 'A Quiet Engine', 'Tides', 'Salt and Weather']


A trailing hyphen is an operator with nothing to exclude. `web_query` dropped it and searched for
the word that was there.


**6.** A search that never raises.


In [7]:
def safe_search(typed, limit=5):
    """Whatever was typed, return a list of books, never an exception."""
    query = BookIndex.web_query(typed or "")
    if not query.strip():
        return []
    try:
        return list(Book.select()
                        .join(BookIndex, on=(Book.id == BookIndex.rowid))
                        .where(BookIndex.match(query))
                        .order_by(BookIndex.bm25())
                        .limit(limit))
    except OperationalError:                                        # anything web_query did not cover
        return []


for entry in ("granite", "c++ OR", "", '""'):
    print(f"  {entry!r:<10} {[book.title for book in safe_search(entry)]}")


  'granite'  ['Stone and Slate']
  'c++ OR'   ['Learning C++']
  ''         []
  '""'       []


The `try` is still there even with `web_query` in front of it, because a translator that handles
everything anybody will ever type is not a thing to bet a page on. An empty list is a result the
page can render, and an exception is not.


---

&#8592; **Back to:** [FTS5Model and SearchField](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/09-fts5model-and-searchfield.ipynb)  &nbsp;&middot;&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
